# 03 — Loại ảnh trùng hoàn toàn

So sánh kích thước và toàn bộ pixel RGB bằng SHA-256; các ảnh có cùng mã được so sánh pixel trực tiếp. Cùng mệnh giá thì giữ bản đầu. Nếu pixel giống nhau nhưng mệnh giá khác, giữ tất cả và ghi vào bảng xung đột. PNG nguồn không bị xóa.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from PIL import Image
from IPython.display import display

cwd = Path.cwd().resolve()
root = cwd if (cwd / "DATA_RAW").is_dir() else cwd.parents[1]
data_crop = root / "DATA_CROP"
metadata_dir = data_crop / "metadata"
input_csv = metadata_dir / "crop_images.csv"
images = pd.read_csv(input_csv, dtype=str).fillna("")
denominations = ["10000", "20000", "50000", "100000", "200000", "500000"]
source_columns = list(images.columns)


In [2]:
groups = {}
for row_number, source_row in enumerate(images.to_dict("records")):
    row = dict(source_row)
    row["candidate_id"] = Path(row["crop_path"]).stem
    row["_row_number"] = row_number
    crop_path = data_crop / row["crop_path"]
    with Image.open(crop_path) as image:
        rgb = image.convert("RGB")
        width, height = rgb.size
        pixel_bytes = rgb.tobytes()
    pixel_sha256 = hashlib.sha256(width.to_bytes(4, "big") + height.to_bytes(4, "big") + pixel_bytes).hexdigest()
    row["pixel_width"] = width
    row["pixel_height"] = height
    row["pixel_sha256"] = pixel_sha256
    key = (width, height, pixel_sha256)
    groups.setdefault(key, []).append(row)

pixel_groups = []
for rows in groups.values():
    if len(rows) == 1:
        pixel_groups.append(rows)
    else:
        verified_groups = []
        for row in rows:
            with Image.open(data_crop / row["crop_path"]) as image:
                pixel_bytes = image.convert("RGB").tobytes()
            matching_group = None
            for group in verified_groups:
                if pixel_bytes == group["pixel_bytes"]:
                    matching_group = group
                    break
            if matching_group is None:
                verified_groups.append({"pixel_bytes": pixel_bytes, "rows": [row]})
            else:
                matching_group["rows"].append(row)
        pixel_groups.extend(group["rows"] for group in verified_groups)


In [3]:
unique_rows = []
duplicate_rows = []
conflict_rows = []

for group in pixel_groups:
    group = sorted(group, key=lambda row: row["_row_number"])
    group_denominations = {row["denomination"] for row in group}
    if len(group_denominations) > 1:
        conflict_rows.extend(group)
        unique_rows.extend(group)
    elif len(group) == 1:
        unique_rows.append(group[0])
    else:
        kept = group[0]
        unique_rows.append(kept)
        for duplicate in group[1:]:
            record = {
                "kept_candidate_id": kept["candidate_id"],
                "kept_dataset_id": kept["dataset_id"],
                "kept_source_rel_path": kept["source_rel_path"],
                "kept_crop_path": kept["crop_path"],
                "duplicate_candidate_id": duplicate["candidate_id"],
                "pixel_width": duplicate["pixel_width"],
                "pixel_height": duplicate["pixel_height"],
                "pixel_sha256": duplicate["pixel_sha256"],
            }
            for column in source_columns:
                record[f"duplicate_{column}"] = duplicate[column]
            duplicate_rows.append(record)

unique_columns = source_columns + ["candidate_id", "pixel_width", "pixel_height", "pixel_sha256"]
unique_rows = sorted(unique_rows, key=lambda row: row["_row_number"])
unique_crops = pd.DataFrame(
    [{column: row[column] for column in unique_columns} for row in unique_rows],
    columns=unique_columns,
)
conflict_rows = sorted(conflict_rows, key=lambda row: row["_row_number"])
label_conflicts = pd.DataFrame(
    [{column: row[column] for column in unique_columns} for row in conflict_rows],
    columns=unique_columns,
)
duplicate_columns = [
    "kept_candidate_id", "kept_dataset_id", "kept_source_rel_path", "kept_crop_path",
    "duplicate_candidate_id", *[f"duplicate_{column}" for column in source_columns],
    "pixel_width", "pixel_height", "pixel_sha256",
]
exact_duplicates = pd.DataFrame(duplicate_rows, columns=duplicate_columns)

unique_crops.to_csv(metadata_dir / "unique_crops.csv", index=False, encoding="utf-8-sig")
exact_duplicates.to_csv(metadata_dir / "exact_duplicates.csv", index=False, encoding="utf-8-sig")
label_conflicts.to_csv(metadata_dir / "label_conflicts.csv", index=False, encoding="utf-8-sig")

before = images["denomination"].value_counts().reindex(denominations, fill_value=0)
after = unique_crops["denomination"].value_counts().reindex(denominations, fill_value=0)
removed = exact_duplicates["duplicate_denomination"].value_counts().reindex(denominations, fill_value=0)
conflicts = label_conflicts["denomination"].value_counts().reindex(denominations, fill_value=0)
summary = pd.DataFrame({"before": before, "unique_after": after, "exact_duplicates_removed": removed, "conflict_rows_kept": conflicts})
summary.index.name = "denomination"
summary.loc["Tổng"] = summary.sum()
display(summary)


,before,unique_after,exact_duplicates_removed,conflict_rows_kept
denomination,,,,
10000,2895,2895,0,0
20000,2378,2378,0,0
50000,2064,2064,0,0
100000,2415,2415,0,0
200000,3151,3151,0,0
500000,2708,2708,0,0
Tổng,15611,15611,0,0
